# 06 — Conditional Flow Matching for Molecular Generation (Novel Algorithm)

**No API key required.** Train a generative model that creates new molecules by integrating a learned vector field conditioned on target properties.

Theory:
- Optimal Transport CFM (Lipman et al., ICLR 2023)
- Molecular-space analogue of SemlaFlow (arxiv:2406.07266)
- 100x faster than DDPM-based molecular generation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from chemvision.core.reproducibility import set_global_seed
set_global_seed(42)

## 6.1 — Prepare training data

In [ ]:
from chemvision.data.dataset_builder import MolecularDatasetBuilder
from pathlib import Path

builder = MolecularDatasetBuilder(seed=42)
builder.add_seeds()
builder.add_random_molecules(n=100)
stats = builder.build(Path('/tmp/flow_data'))
fps, props, splits = MolecularDatasetBuilder.load_arrays('/tmp/flow_data')
print(f'Dataset: {fps.shape[0]} molecules, {fps.shape[1]}-dim fingerprints, {props.shape[1]} properties')

## 6.2 — Train the flow matcher

In [ ]:
from chemvision.generation._experimental.flow_matcher import ConditionalFlowMatcher, FlowMatcherConfig

config = FlowMatcherConfig(
    fp_dim=fps.shape[1],
    cond_dim=props.shape[1],
    hidden_dim=256,
    n_layers=3,
    learning_rate=5e-4,
    seed=42,
)

cfm = ConditionalFlowMatcher(config)
result = cfm.train(fps, props, epochs=80, batch_size=32, patience=25, verbose=True)

print(f'\nTraining complete:')
print(f'  Epochs: {result.epochs}')
print(f'  Final loss: {result.final_loss:.4f}')
print(f'  Converged: {result.converged}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(result.loss_history, color='darkorange', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (vector field)')
ax.set_title(f'Flow Matcher Training — {result.epochs} epochs')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6.3 — Generate new molecules conditioned on target properties

In [ ]:
# Define target property profiles
# Columns: [MW, LogP, TPSA, HBD, HBA, RotBonds, Rings, QED]
targets = np.array([
    [180, 1.2, 60, 1, 4, 3, 1, 0.55],   # aspirin-like
    [150, 0.5, 40, 2, 2, 1, 1, 0.70],   # small drug-like
    [300, 3.0, 80, 1, 5, 5, 2, 0.45],   # larger drug
    [120, -0.5, 30, 1, 1, 0, 0, 0.80],  # very small, high QED
], dtype=np.float32)

generated = cfm.sample(targets, n_steps=30)

for i, g in enumerate(generated):
    n_bits = int(g.binary_fingerprint.sum())
    print(f'Target {i}: MW={targets[i,0]:.0f}, QED={targets[i,7]:.2f} -> '
          f'Generated FP with {n_bits} bits set')

## 6.4 — Compare generated vs real fingerprints

In [ ]:
# Compare bit density: real vs generated
real_density = [fp.sum() for fp in fps]
gen_density = [g.binary_fingerprint.sum() for g in generated]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(real_density, bins=20, color='steelblue', edgecolor='navy', alpha=0.8)
axes[0].set_xlabel('Non-zero bits')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Real molecules (n={len(fps)})')

# Show generated as vertical lines on the real distribution
for d in gen_density:
    axes[0].axvline(d, color='red', linestyle='--', alpha=0.6)
axes[0].legend(['Generated'] + [''] * (len(gen_density)-1), loc='upper right')

# Fingerprint heatmap comparison
axes[1].imshow(
    np.vstack([fps[0].reshape(1, -1)[:, :256], generated[0].binary_fingerprint.reshape(1, -1)[:, :256]]),
    aspect='auto', cmap='Blues', interpolation='nearest'
)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Real', 'Generated'])
axes[1].set_xlabel('Bit index (first 256)')
axes[1].set_title('Fingerprint comparison')

plt.tight_layout()
plt.show()

## 6.5 — Nearest-neighbour lookup of generated fingerprints

In [ ]:
from chemvision.retrieval.vector_store import MoleculeVectorStore

# Build a store of all real molecules
store = MoleculeVectorStore()
train_df = pd.read_parquet('/tmp/flow_data/train.parquet')

for i in range(len(fps)):
    smi = train_df.iloc[i]['smiles'] if i < len(train_df) else f'mol_{i}'
    store.add(str(smi), fps[i])

# For each generated fingerprint, find the nearest real molecule
print('Nearest real molecule to each generated fingerprint:')
print('=' * 60)
for i, g in enumerate(generated):
    hits = store.search(g.binary_fingerprint, k=3)
    print(f'\nGenerated {i} (target MW={targets[i,0]:.0f}):')
    for h in hits:
        print(f'  {h["name"][:50]}  (similarity={h["score"]:.3f})')